# Step 2 — Grid and Boundary Setup

Before solving the PDE we need:
1. A stock price grid S and a time grid t
2. The terminal payoff at expiry
3. The boundary conditions at S=0 and S=S_max

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.grid import make_stock_grid, make_time_grid
from src.payoff import call_payoff, put_payoff
from src.boundary_conditions import call_boundaries, put_boundaries

## Parameters

In [ ]:
K     = 100
r     = 0.05
sigma = 0.20
T     = 1.0
S_max = 3 * K

M = 10   # stock grid points (small so we can inspect the values)
N = 5    # time steps

## 1. Stock and Time Grids

In [ ]:
S = make_stock_grid(S_max, M)
t = make_time_grid(T, N)

print('Stock grid S:', np.round(S, 2))
print('Time grid  t:', np.round(t, 3))

## 2. Terminal Payoff at t = T

In [ ]:
payoff_call = call_payoff(S, K)
payoff_put  = put_payoff(S, K)

plt.figure(figsize=(8, 3.5))
plt.plot(S, payoff_call, label='Call payoff  max(S-K, 0)', lw=2)
plt.plot(S, payoff_put,  label='Put payoff   max(K-S, 0)', lw=2, ls='--')
plt.axvline(K, color='grey', ls=':', label=f'Strike K={K}')
plt.xlabel('Stock price S')
plt.ylabel('Payoff')
plt.title('Terminal payoff at t = T')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Boundary Conditions over Time

We evaluate the boundary values at each time step to see how they evolve.

- **S = 0**: call = 0 always; put = K·e^{-rτ} (present value of strike)
- **S = S_max**: call = S_max - K·e^{-rτ}; put = 0 always

In [ ]:
taus = np.linspace(0, T, 100)   # time-to-expiry from 0 (expiry) to T (now)

lower_call = [call_boundaries(S_max, K, r, tau)[0] for tau in taus]
upper_call = [call_boundaries(S_max, K, r, tau)[1] for tau in taus]
lower_put  = [put_boundaries(S_max, K, r, tau)[0]  for tau in taus]
upper_put  = [put_boundaries(S_max, K, r, tau)[1]  for tau in taus]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(taus, lower_call, label='V(0, τ) = 0')
axes[0].plot(taus, upper_call, label=f'V(S_max, τ) = S_max - K·e^(-rτ)')
axes[0].set_xlabel('Time to expiry τ')
axes[0].set_ylabel('Boundary value')
axes[0].set_title('Call boundary conditions')
axes[0].legend(fontsize=8)

axes[1].plot(taus, lower_put, label='V(0, τ) = K·e^(-rτ)')
axes[1].plot(taus, upper_put, label='V(S_max, τ) = 0')
axes[1].set_xlabel('Time to expiry τ')
axes[1].set_title('Put boundary conditions')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Visualising the Full Grid

Each cell in this mesh will hold an option value. We start by filling in the terminal row (payoff at t=T) and the two boundary columns.

In [ ]:
# Build a grid of NaN and fill in what we know — for a call
grid = np.full((M + 1, N + 1), np.nan)

# Terminal condition (last column, tau=0)
grid[:, -1] = call_payoff(S, K)

# Boundary conditions at each time step
for j, t_j in enumerate(t):
    tau = T - t_j
    lo, hi = call_boundaries(S_max, K, r, tau)
    grid[0,  j] = lo
    grid[-1, j] = hi

plt.figure(figsize=(8, 4))
plt.imshow(grid, aspect='auto', origin='lower',
           extent=[t[0], t[-1], S[0], S[-1]])
plt.colorbar(label='Known option value')
plt.xlabel('Time t')
plt.ylabel('Stock price S')
plt.title('Call option grid — known values (interior = NaN = to be solved)')
plt.tight_layout()
plt.show()